# PoliMillionaire Game Tests

Run live games with selected local models, one after another.

No generated-answer API is used.


## Setup

Find the repo and local model cache.


In [1]:
import gc
import os
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "src" / "polimillionaire" / "__init__.py").exists():
            return candidate
    raise RuntimeError("Open this notebook from inside the project folder.")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
for path in [REPO_ROOT / "src", REPO_ROOT]:
    path_text = str(path)
    if path_text not in sys.path:
        sys.path.insert(0, path_text)

os.environ.setdefault("HF_HOME", str(REPO_ROOT / "data" / "hf_home"))
os.environ.setdefault("HUGGINGFACE_HUB_CACHE", str(REPO_ROOT / "data" / "hf_home" / "hub"))

print("Repo:", REPO_ROOT)
print("HF cache:", os.environ["HF_HOME"])


Repo: /Users/camiloa2m/Downloads/NLP/nlp-polimillionaire
HF cache: /Users/camiloa2m/Downloads/NLP/nlp-polimillionaire/data/hf_home


## Settings

Pick models here. Each selected model gets its own game run.


In [2]:
RUN_API_CHECK = True
RUN_LIVE_GAME = True
PROMPT_FOR_CREDENTIALS = False

API_URL = "http://131.175.15.22:51111/"
COMPETITION_ID = 0

MODELS_TO_RUN = [
    {"label": "Gemma 4 E2B", "model_id": "google/gemma-4-E2B-it", "run": False},
    {"label": "Gemma 4 E4B", "model_id": "google/gemma-4-E4B-it", "run": False},
    {"label": "Qwen 2.5 3B", "model_id": "Qwen/Qwen2.5-3B-Instruct", "run": False},
    {"label": "Qwen 2.5 7B", "model_id": "Qwen/Qwen2.5-7B-Instruct", "run": True},
]

SAFE_DELAY_SECONDS = 2.0
ANSWER_TIMEOUT_SECONDS = 25.0

print("API check:", RUN_API_CHECK)
print("Live game:", RUN_LIVE_GAME)
print("Competition:", COMPETITION_ID)
print("Models:")
for item in MODELS_TO_RUN:
    print("-", item["label"], item["model_id"], "run=", item["run"])


API check: True
Live game: True
Competition: 0
Models:
- Gemma 4 E2B google/gemma-4-E2B-it run= False
- Gemma 4 E4B google/gemma-4-E4B-it run= False
- Qwen 2.5 3B Qwen/Qwen2.5-3B-Instruct run= False
- Qwen 2.5 7B Qwen/Qwen2.5-7B-Instruct run= True


## Login

Use environment variables, Colab secrets, or prompt mode.


In [3]:
import getpass
from dotenv import load_dotenv
from millionaire_client import AuthenticationError, MillionaireClient

# Load environment variables from .env file
load_dotenv()

USERNAME = os.environ.get("POLIMILLIONAIRE_USERNAME")
PASSWORD = os.environ.get("POLIMILLIONAIRE_PASSWORD")

try:
    from google.colab import userdata

    USERNAME = USERNAME or userdata.get("POLIMILLIONAIRE_USERNAME")
    PASSWORD = PASSWORD or userdata.get("POLIMILLIONAIRE_PASSWORD")
except Exception:
    pass

if PROMPT_FOR_CREDENTIALS and not USERNAME:
    USERNAME = input("PoliMillionaire username: ").strip()
if PROMPT_FOR_CREDENTIALS and not PASSWORD:
    PASSWORD = getpass.getpass("PoliMillionaire password: ")

print("Username configured:", bool(USERNAME))
print("Password configured:", bool(PASSWORD))


Username configured: True
Password configured: True


## Competitions

Turn on `RUN_API_CHECK` to list games.


In [4]:
client = None

if RUN_API_CHECK:
    if not USERNAME or not PASSWORD:
        raise RuntimeError("Set credentials first.")
    try:
        client = MillionaireClient(API_URL)
        user = client.login(USERNAME, PASSWORD)
        print("Logged in as", user.username)
        for competition in client.competitions.list_all():
            print(competition.id, competition.name, competition.max_levels)
    except AuthenticationError as exc:
        client = None
        print("Login failed:", exc)
else:
    print("API check skipped.")


Logged in as promptPotamo
0 Entertainment 15
1 Ancient History and Politics 15
2 Science and Nature 15
3 Maths 15
4 Philosophy and Psychology 15
5 News 15


## Run Selected Models

Each model warms up, plays if enabled, then unloads.


In [5]:
# !pip install  -qU colab-xterm
# %load_ext colabxterm
# # It may ask you to restart, accept and run this cell again

### For Colab: Run the following commands on the terminal below:
To spawn the server deamon please run the cell with ```%xterm```. It will create a concurrent shell where we can paste the following commands but, at the same time, also run other cells!

```sudo apt-get install zstd```

```curl -fsSL https://ollama.com/install.sh | sh```

```ollama serve & ollama pull llama3```

In [6]:
# %xterm

In [7]:
from dataclasses import dataclass, field
from typing import Any

"""Small dataclasses shared by the assignment notebook helpers."""

@dataclass(frozen=True)
class AnswerOption:
    id: int
    text: str


@dataclass(frozen=True)
class Question:
    id: int
    text: str
    options: list[AnswerOption]
    level: int = 0
    metadata: dict[str, Any] = field(default_factory=dict)

    def valid_option_ids(self) -> set[int]:
        return {option.id for option in self.options}

    def first_option(self) -> AnswerOption:
        if not self.options:
            raise ValueError("Question has no answer options")
        return self.options[0]

    def get_option(self, option_id: int) -> AnswerOption | None:
        for option in self.options:
            if option.id == option_id:
                return option
        return None

    def require_option(self, option_id: int) -> AnswerOption:
        return self.get_option(option_id) or self.first_option()


@dataclass
class AnswerPrediction:
    option_id: int
    answer_text: str
    confidence: float | None = None
    reasoning: str | None = None
    metadata: dict[str, Any] = field(default_factory=dict)



In [ ]:
# ============================================================
# INSTALL (run once if needed)
# ============================================================

# !pip install -q \
#     langchain \
#     langchain-core \
#     langchain-community \
#     langchain-ollama \
#     ddgs \
#     numexpr \
#     pydantic

# ============================================================
# IMPORTS
# ============================================================

import math
import logging
from enum import Enum
from typing import Literal

import numexpr as ne

from ddgs import DDGS

from pydantic import BaseModel, Field

from langchain_ollama import ChatOllama

from langchain_core.tools import tool
from langchain_core.messages import (
    HumanMessage,
    SystemMessage,
    ToolMessage
)
from langchain_core.prompts import ChatPromptTemplate

from langchain_core.runnables import (
    RunnableLambda,
    RunnableBranch
)

# ============================================================
# LOGGING
# ============================================================

logging.basicConfig(level=logging.INFO)
log = logging.getLogger("trivia-agent")

# ============================================================
# CONFIG
# ============================================================

MODEL_NAME = "qwen2.5:7b-instruct-q4_K_M"

TEMPERATURE = 0

MAX_RESULTS_FACTUAL = 3
MAX_RESULTS_RECENT = 5

TIMEOUT_DDG = 10

# ============================================================
# QUESTION STRUCTURES
# ============================================================

from dataclasses import dataclass

@dataclass(frozen=True)
class AnswerOption:
    id: int
    text: str


@dataclass(frozen=True)
class Question:
    id: int
    text: str
    options: list[AnswerOption]


# ============================================================
# OUTPUT SCHEMA
# ============================================================

class TriviaAnswer(BaseModel):

    answer: int = Field(
        description="Selected option ID"
    )

    confidence: float = Field(
        ge=0.0,
        le=1.0,
        description="Confidence score"
    )

    evidence: str = Field(
        description="Short factual justification"
    )

# ============================================================
# ROUTING
# ============================================================

class Route(str, Enum):
    CALCULATION = "calculation"
    FACTUAL = "factual"
    RECENT = "recent"


class RouteDecision(BaseModel):
    route: Route


# ============================================================
# LLM
# ============================================================

llm = ChatOllama(
    model=MODEL_NAME,
    temperature=TEMPERATURE,
)

structured_llm = llm.with_structured_output(
    TriviaAnswer
)

router_llm = llm.with_structured_output(
    RouteDecision
)

# ============================================================
# TOOLS
# ============================================================

@tool
def duckduckgo_search(query: str) -> str:
    """
    Search recent or changing information from the web.
    """

    try:

        with DDGS(timeout=TIMEOUT_DDG) as ddgs:

            results = ddgs.text(
                query,
                max_results=MAX_RESULTS_RECENT,
                backend="duckduckgo"
            )

            results = list(results)

            if not results:
                return "No results found."

            return "\n\n".join(
                f"TITLE: {r.get('title', '')}\n"
                f"SNIPPET: {r.get('body', '')}"
                for r in results
            )

    except Exception as e:
        return f"Search error: {e}"


@tool
def calculator(expression: str) -> str:
    """
    Evaluate a mathematical expression.

    Supports:
    + - * / **
    sqrt()
    sin()
    cos()
    tan()

    IMPORTANT:
    Trigonometric functions use radians.
    """

    try:

        result = ne.evaluate(
            expression,
            local_dict={
                "pi": math.pi,
                "e": math.e,
            }
        )

        result = result.item()

        if isinstance(result, float):
            result = round(result, 6)

        return str(result)

    except Exception as e:
        return f"Calculator error: {e}"

# ============================================================
# TOOL-AWARE RETRIEVAL MODEL
# ============================================================

retrieval_llm = llm.bind_tools([
    duckduckgo_search
])

# ============================================================
# ROUTER PROMPT
# ============================================================

ROUTER_PROMPT = """
Classify the question into ONE category:

1. CALCULATION
Use ONLY if the question requires:
- arithmetic
- numeric computation
- algebraic solving
- evaluating expressions

2. RECENT
Use for:
- recent events
- changing information
- elections
- sports winners
- current leaders
- news
- questions involving years like 2024, 2025, 2026

3. FACTUAL
Use for:
- definitions
- historical facts
- scientific concepts
- questions ABOUT formulas
- conceptual explanations
- stable knowledge

IMPORTANT:
Questions ABOUT mathematics are FACTUAL.
Questions REQUIRING computation are CALCULATION.

Examples:

Q: What is 25% of 320?
Route: CALCULATION

Q: Solve 5x + 2 = 12
Route: CALCULATION

Q: What is the formula for the volume of a cone?
Route: FACTUAL

Q: What is the Pythagorean theorem?
Route: FACTUAL

Q: Who won the UEFA Champions League in 2025?
Route: RECENT
"""

# ============================================================
# ROUTER
# ============================================================

def route_question(question: str) -> Route:

    result = router_llm.invoke([
        SystemMessage(content=ROUTER_PROMPT),
        HumanMessage(content=question)
    ])

    return result.route

# ============================================================
# FACTUAL RETRIEVAL (Wikipedia backend)
# ============================================================

def factual_retrieval(question: str) -> str:

    try:

        with DDGS(timeout=TIMEOUT_DDG) as ddgs:

            results = ddgs.text(
                question,
                max_results=MAX_RESULTS_FACTUAL,
                backend="wikipedia"
            )

            results = list(results)

            if not results:
                return "No factual evidence found."

            return "\n\n".join(
                f"TITLE: {r.get('title', '')}\n"
                f"SNIPPET: {r.get('body', '')}"
                for r in results
            )

    except Exception as e:
        return f"Wikipedia backend error: {e}"

# ============================================================
# RECENT RETRIEVAL (AUTOMATIC TOOL CALLING)
# ============================================================

def recent_retrieval(question: str) -> str:

    messages = [
        SystemMessage(
            content="""
You are a retrieval assistant.

You MUST use duckduckgo_search
for recent or changing information.
"""
        ),
        HumanMessage(content=question)
    ]

    response = retrieval_llm.invoke(messages)

    tool_outputs = []

    if response.tool_calls:

        for tc in response.tool_calls:

            tool_name = tc["name"]
            tool_args = tc["args"]

            if tool_name == "duckduckgo_search":

                result = duckduckgo_search.invoke(tool_args)

                tool_outputs.append(result)

                messages.append(response)

                messages.append(
                    ToolMessage(
                        content=result,
                        tool_call_id=tc["id"]
                    )
                )

    return "\n\n".join(tool_outputs)

# ============================================================
# MATH PIPELINE
# ============================================================

def extract_expression(question: str) -> str:

    prompt = f"""
Extract ONLY the mathematical expression.

IMPORTANT:
- Convert degrees to radians if needed
- Use pi for π
- Return ONLY the expression

Examples:

Question:
What is cosine of 90 degrees?

Answer:
cos(pi/2)

Question:
What is 25% of 320?

Answer:
0.25 * 320

Question:
{question}
"""

    return llm.invoke(prompt).content.strip()


def run_math(question: str) -> str:

    expression = extract_expression(question)

    log.info(f"Expression: {expression}")

    result = calculator.invoke({
        "expression": expression
    })

    return (
        f"Expression: {expression}\n"
        f"Result: {result}"
    )

# ============================================================
# MCQ FORMATTER
# ============================================================

def build_mcq(question: Question) -> str:

    options = "\n".join(
        f"{o.id}. {o.text}"
        for o in question.options
    )

    return (
        f"{question.text}\n\n"
        f"{options}"
    )

# ============================================================
# REASONING
# ============================================================

SYSTEM_PROMPT = """
You are a precise multiple-choice trivia assistant.

Rules:
1. Prioritize retrieved evidence over memorized knowledge
2. Select EXACTLY ONE option ID
3. Keep reasoning concise and factual
4. Lower confidence if evidence is weak
5. Confidence must be between 0.0 and 1.0
6. Never invent evidence
"""

REASONING_PROMPT = """
QUESTION:
{question}

EVIDENCE:
{evidence}

Pick the correct option ID.
"""

reasoning_prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", REASONING_PROMPT)
])

reasoning_chain = reasoning_prompt | structured_llm

# ============================================================
# PIPELINES
# ============================================================

def calculation_pipeline(question: Question) -> str:
    return run_math(question.text)


def factual_pipeline(question: Question) -> str:
    return factual_retrieval(build_mcq(question))


def recent_pipeline(question: Question) -> str:
    return recent_retrieval(build_mcq(question))

# ============================================================
# RUNNABLE ROUTING PIPELINE
# ============================================================

routing_chain = RunnableLambda(
    lambda q: {
        "question": q,
        "route": route_question(q.text)
    }
)

pipeline = routing_chain | RunnableBranch(

    (
        lambda x: x["route"] == Route.CALCULATION,
        RunnableLambda(
            lambda x: {
                "question": x["question"],
                "evidence": calculation_pipeline(
                    x["question"]
                )
            }
        )
    ),

    (
        lambda x: x["route"] == Route.RECENT,
        RunnableLambda(
            lambda x: {
                "question": x["question"],
                "evidence": recent_pipeline(
                    x["question"]
                )
            }
        )
    ),

    RunnableLambda(
        lambda x: {
            "question": x["question"],
            "evidence": factual_pipeline(
                x["question"]
            )
        }
    )
)

# ============================================================
# MAIN ENTRYPOINT
# ============================================================

def ask_question(question: Question):

    log.info("=" * 80)
    log.info("QUESTION")
    log.info("=" * 80)

    mcq = build_mcq(question)

    log.info(mcq)

    result = pipeline.invoke(question)

    evidence = result["evidence"]

    log.info("=" * 80)
    log.info("EVIDENCE")
    log.info("=" * 80)

    log.info(evidence[:2000])

    final_answer = reasoning_chain.invoke({
        "question": mcq,
        "evidence": evidence
    })

    log.info("=" * 80)
    log.info("FINAL ANSWER")
    log.info("=" * 80)

    log.info(
        final_answer.model_dump_json(indent=2)
    )

    return final_answer

# ============================================================
# EXAMPLE
# ============================================================

question = Question(
    id=1,
    text="Which country won the FIFA World Cup in 2018?",
    options=[
        AnswerOption(0, "Germany"),
        AnswerOption(1, "Argentina"),
        AnswerOption(2, "France"),
        AnswerOption(3, "Brazil"),
    ],
)

answer = ask_question(question)

print(answer.model_dump_json(indent=2))

INFO:trivia-agent:=== QUESTION ===
INFO:trivia-agent:What is the primary reason Mariah Carey was criticized for her performance on 'Dick Clark's New Year's Rockin' Eve' in 2017?
0. She had technical difficulties with her in-ear monitors
1. She lip-synced throughout the performance
2. She performed an inappropriately inappropriate song
3. She used auto-tune extensively
INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
INFO:trivia-agent:ROUTE: Route.RETRIEVAL
INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
INFO:trivia-agent:DDG query: Mariah Carey criticism 2017 New Year's Rockin' Eve performance lip-syncing
INFO:httpx:HTTP Request: POST https://html.duckduckgo.com/html/ "HTTP/2 200 OK"
INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
INFO:trivia-agent:=== EVIDENCE ===
INFO:trivia-agent:- Mariah Carey performed during New Year's Eve in Times Square, New York.
- Her 2016 New Year's Eve performance

In [21]:
from abc import ABC, abstractmethod
from typing import Any, Protocol
from src.polimillionaire.strategies import parse_answer_prediction

class BaseStrategy(ABC):
    name = "base"

    @abstractmethod
    def answer(self, question: Question) -> AnswerPrediction:
        raise NotImplementedError
    
class LocalLLM(Protocol):
    model_name: str

    def generate(self, prompt: str, **kwargs: object) -> str:
        ...

class OllamaStrategy(BaseStrategy):
    name = "Ollama-Gwen2.5-7b"

    def __init__(self):
        self.llm = ChatOllama(model="gwen2.5:7b", temperature=0)

    def answer(self, question: Question) -> AnswerPrediction:
        prediction = ask_question(question)

        # raw_text = self.llm.generate(build_prompt(question))
        prediction = parse_answer_prediction(str(prediction), question, strategy_name=self.name)
        prediction.metadata["strategy"] = self.name
        prediction.metadata["model_name"] = getattr(self.llm, "model_name", "unknown")
        prediction.metadata["device"] = getattr(self.llm, "device_summary", "unknown")
        return prediction

In [22]:
from datetime import datetime, timezone

from polimillionaire.runner import GameRunner, RunLogger
from polimillionaire.strategies import GemmaLLMConfig, GemmaStrategy
from polimillionaire.types import AnswerOption, Question

warmup_question = Question(
    id=1,
    text="What is 2 + 2?",
    options=[
        AnswerOption(0, "3"),
        AnswerOption(1, "4"),
        AnswerOption(2, "5"),
        AnswerOption(3, "22"),
    ],
)


def clear_model_memory():
    gc.collect()
    try:
        import torch

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
        if hasattr(torch, "mps") and hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
            torch.mps.empty_cache()
    except Exception as exc:
        print("Cleanup warning:", exc)


def make_strategy(model_id: str) -> GemmaStrategy:
    config = GemmaLLMConfig(
        model_id=model_id,
        inference_backend="auto_model",
        device_map="auto",
        dtype="auto",
        max_new_tokens=8,
        temperature=0.0,
        do_sample=False,
        num_beams=1,
        seed=42,
        generation_max_time_seconds=18.0,
        timeout_seconds=120.0,
    )
    return GemmaStrategy(model_config=config)


def clean_name(label: str) -> str:
    return "_".join(label.lower().split())


if RUN_LIVE_GAME and (not USERNAME or not PASSWORD):
    raise RuntimeError("Set credentials first.")
if RUN_LIVE_GAME and client is None:
    client = MillionaireClient(API_URL)
    client.login(USERNAME, PASSWORD)

run_results = []
for item in MODELS_TO_RUN:
    if not item.get("run", False):
        print("Skipped", item["label"])
        continue

    strategy = None
    try:
        print()
        print("Model:", item["label"])
        # strategy = make_strategy(item["model_id"])
        strategy = OllamaStrategy()

        warmup = strategy.answer(warmup_question)
        
        print("warmup option:", warmup.option_id, warmup.answer_text)
        print("fallback:", warmup.metadata.get("fallback"))
        print("device:", warmup.metadata.get("device"))
        if warmup.metadata.get("fallback"):
            raise RuntimeError(f"Warm-up failed for {item['label']}.")

        result = {"label": item["label"], "model_id": item["model_id"], "warmup_option": warmup.option_id}

        if RUN_LIVE_GAME:
            run_id = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
            log_path = REPO_ROOT / "results" / "runs" / f"{clean_name(item['label'])}_{run_id}.jsonl"
            runner = GameRunner(
                client=client,
                safe_delay_seconds=SAFE_DELAY_SECONDS,
                answer_timeout_seconds=ANSWER_TIMEOUT_SECONDS,
                logger=RunLogger(log_path),
            )
            game = runner.play(COMPETITION_ID, strategy)
            result.update(
                {
                    "current_level": game.current_level,
                    "earned_amount": game.earned_amount,
                    "log_path": str(log_path),
                }
            )
            print("Reached level:", game.current_level)
            print("Earned amount:", game.earned_amount)
            print("Log path:", log_path)
        else:
            print("Live game skipped for", item["label"])

        run_results.append(result)
    finally:
        del strategy
        clear_model_memory()
        print("Cleared model memory after", item["label"])

run_results


INFO:trivia-agent:=== QUESTION ===
INFO:trivia-agent:What is 2 + 2?
0. 3
1. 4
2. 5
3. 22


Skipped Gemma 4 E2B
Skipped Gemma 4 E4B
Skipped Qwen 2.5 3B

Model: Qwen 2.5 7B


INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
INFO:trivia-agent:ROUTE: Route.CALCULATION
INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
INFO:trivia-agent:Math expression: 2 + 2
INFO:trivia-agent:=== EVIDENCE ===
INFO:trivia-agent:Expression: 2 + 2
Result: 4
INFO:trivia-agent:=== RESULT ===
INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
INFO:trivia-agent:answer=1 confidence=1.0 evidence='Expression: 2 + 2\nResult: 4'
INFO:trivia-agent:=== RESULT (STRUCTURED) ===
INFO:trivia-agent:{
  "answer": 1,
  "confidence": 1.0,
  "evidence": "Expression: 2 + 2\nResult: 4"
}
INFO:trivia-agent:=== QUESTION ===
INFO:trivia-agent:How does the film The Babadook relate to Jennifer Kent's previous work?
0. It is a short film she directed
1. It is a remake of her previous film
2. It is an extension of her short film Monster
3. It is unrelated to her previous works


warmup option: 1 4
fallback: False
device: unknown


INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
INFO:trivia-agent:ROUTE: Route.RETRIEVAL
INFO:trivia-agent:DDG query: How does the film The Babadook relate to Jennifer Kent's previous work?
INFO:primp:response: https://grokipedia.com/api/typeahead?query=How+does+the+film+The+Babadook+relate+to+Jennifer+Kent%27s+previous+work%3F&limit=1 200
INFO:primp:response: https://en.wikipedia.org/w/api.php?action=opensearch&profile=fuzzy&limit=1&search=How%20does%20the%20film%20The%20Babadook%20relate%20to%20Jennifer%20Kent%27s%20previous%20work%3F 200
INFO:primp:response: https://www.mojeek.com/search?q=How+does+the+film+The+Babadook+relate+to+Jennifer+Kent%27s+previous+work%3F 200
INFO:trivia-agent:=== EVIDENCE ===
INFO:trivia-agent:TITLE: After The Babadook: Jennifer Kent’s new film tackles
SNIPPET: And with the runaway success of horror-doused psychological thriller The Babadook, director-writer Jennifer Kent’s 2014 debut feature film, it ...

TITLE: Jennifer Ke

Reached level: 1
Earned amount: 0
Log path: /Users/camiloa2m/Downloads/NLP/nlp-polimillionaire/results/runs/qwen_2.5_7b_20260526_095129.jsonl
Cleared model memory after Qwen 2.5 7B


[{'label': 'Qwen 2.5 7B',
  'model_id': 'Qwen/Qwen2.5-7B-Instruct',
  'warmup_option': 1,
  'current_level': 1,
  'earned_amount': 0,
  'log_path': '/Users/camiloa2m/Downloads/NLP/nlp-polimillionaire/results/runs/qwen_2.5_7b_20260526_095129.jsonl'}]

## Results

Read recent game logs.


In [ ]:
from polimillionaire.runner import load_jsonl, summarize_attempts

run_logs = sorted((REPO_ROOT / "results" / "runs").glob("*.jsonl"), key=lambda path: path.stat().st_mtime)

for path in run_logs[-5:]:
    print(path.name)

if run_logs:
    rows = load_jsonl(run_logs[-1])
    print("latest:", run_logs[-1].name)
    print(summarize_attempts(rows))
else:
    print("No logs found.")


## Done

Use `run_results` and the JSONL logs for comparison.
